In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score,f1_score,confusion_matrix

week7_record = {
    "params": {"max_depth": 3, "min_samples_leaf": 1, "class_weight": "balanced"},
    "random_state": 42, "threshold": 0.5,
    "test_recall": 0.745, "test_f1": 0.536,
    "found": 41, "missed": 14, "false_alarms": 57,
}
df = pd.read_csv(r"C:\Users\Ajvad\Desktop\5th sem ojt\project 1_diabaties\data\processed\muhammed_ajvad_sinan_p1w2_features.csv")

feature_cols = [
    "age", "pregnancies", "glucose", "blood_pressure",
    "skin_thickness", "insulin", "bmi", "diabetes_pedigree",
    "bmi_category", "glucose_band"
]

x= pd.get_dummies(df[feature_cols],drop_first=True)
y=df["outcome"]

x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)

final_model = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    max_depth=3,
    min_samples_leaf=1
)

final_model.fit(x_train, y_train)
print("Final Random Forest trained.")

pred = final_model.predict(x_test)

proba = final_model.predict_proba(x_test)[:, 1]

cm = confusion_matrix(y_test, pred)
tn, fp, fn, tp = cm.ravel()

test_recall = recall_score(y_test, pred, zero_division=0)
test_f1 = f1_score(y_test, pred, zero_division=0)


print("Test recall:", round(test_recall, 3))
print("Test F1:", round(test_f1, 3))

print(f"found {tp} | missed {fn} | false alarms {fp}")


Final Random Forest trained.
Test recall: 0.745
Test F1: 0.536
found 41 | missed 14 | false alarms 57


## What does the moedel relay on, and does it make it medical sense ?

#### Feature Importance
Feature importance measures how much each feature contributes to a machine learning model's predictions. Higher importance means the feature has a greater influence on the model's decisions.

#### Permutation Importance
Permutation importance measures how much a model's performance decreases when the values of one feature are randomly shuffled. A bigger performance drop indicates that the feature is more important.

#### Difference
- Feature Importance: Measures a feature's contribution to the model.
- Permutation Importance: Measures the decrease in model performance after shuffling a feature.

In [3]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(final_model, x_test, y_test, scoring='recall',n_repeats=10, random_state=42)
both = pd.DataFrame({'built_in'  : final_model.feature_importances_,
                     'perm_mean' : pi.importances_mean,
                     'perm_std'  : pi.importances_std}, index =x.columns)
both['rank_built_in'] = both['built_in'].rank(ascending=False).astype(int)
both['rank_perm']     = both['perm_mean'].rank(ascending=False).astype(int)
print(both.sort_values('perm_mean',ascending=False).round(3).to_string())

print('\nshare of built-in importance held by top three:',round(both['built_in'].nlargest(3).sum(),3))

                            built_in  perm_mean  perm_std  rank_built_in  rank_perm
glucose                        0.323      0.189     0.039              1          1
glucose_band_Normal            0.148      0.113     0.030              2          2
blood_pressure                 0.031      0.022     0.011              8          3
diabetes_pedigree              0.099      0.018     0.028              5          4
skin_thickness                 0.036      0.016     0.015              7          5
insulin                        0.047      0.016     0.015              6          5
bmi_category_Over-weight       0.011      0.013     0.008             11          7
age                            0.135      0.002     0.022              3          8
bmi_category_Under-weight      0.003      0.000     0.000             13         10
glucose_band_Prediabetic       0.008      0.000     0.000             12         10
bmi                            0.111      0.000     0.036              4    

## Count the callbacks 

In [5]:
print(f"Flagged {pred.sum()} of {len(pred)}="
       f"({pred.mean() * 100:.1f}%) patients screened")
yt = y_test.values
rows = []
for t in [0.40, 0.45, 0.50, 0.55, 0.60]:
  p = (proba >= t).astype(int)
  rows.append({
        'threshold': t, 
        'flagged_per_100': round(p.mean() * 100, 1), 
        'found': int(((p == 1) & (y_test == 1)).sum()), 
        'missed': int(((p == 0) & (y_test == 1)).sum()), 
        'healthy_called_back': int(((p == 1) & (y_test == 0)).sum())})

menu = pd.DataFrame(rows)
print(menu.to_string(index=False))

# how many patients get a borderline score, and how diabetic are they 
mid = (proba >= 0.40) & (proba >= 0.60)

print(f'\nscore between 0.40 and 0.60: {mid.sum()} of {len(proba)} patients '
      f'({mid.mean():.0%}); diabetic among them: {yt[mid].mean():.0%}'
      f'(overall: {yt.mean():.0%})')

Flagged 98 of 190=(51.6%) patients screened
 threshold  flagged_per_100  found  missed  healthy_called_back
      0.40             73.7     52       3                   88
      0.45             65.3     48       7                   76
      0.50             51.6     41      14                   57
      0.55             30.0     27      28                   30
      0.60             11.1     13      42                    8

score between 0.40 and 0.60: 21 of 190 patients (11%); diabetic among them: 62%(overall: 29%)


### Find the person the model fails 

In [ ]:
test_df = df.loc[x_test.index].copy()
test_df['flagged'] = pred  

for col in ['age_band', 'bmi_category', 'glucose_band']:
    rows = []
    for value, sub in test_df.groupby(col):
        pos = int((sub['outcome'] == 1).sum())
        hit = int(((sub['outcome'] == 1) & (sub['flagged'] == 1)).sum())
        rows.append({col: value, 'diabetic': pos, 'found': hit, 'missed': pos- hit, 'recall': round(hit/pos, 2) if pos else None})
    print(pd.DataFrame(rows).to_string(index=False), '\n')

   age_band  diabetic  found  missed  recall
Middle-Aged        14     11       3    0.79
     Senior        27     22       5    0.81
      Youth        14      8       6    0.57 

 bmi_category  diabetic  found  missed  recall
       Normal         7      5       2    0.71
        Obese        32     28       4    0.88
  Over-weight        12      6       6    0.50
Under-weight          4      2       2    0.50 

glucose_band  diabetic  found  missed  recall
    Diabetic        30     28       2    0.93
     Normal          4      0       4    0.00
 Prediabetic        21     13       8    0.62 

